# 1D WYYKK Kernel Integral Demo

Small executable demo for the internal B-spline order convention. The public integer API is unchanged: `-1` means direct/indicator, `0,1,2,...` mean B-spline order. Internally these are represented as `kind=:indicator` or `kind=:bspline`.


## 1. Setup


In [ ]:
using Pkg
repo_root = normpath(joinpath(@__DIR__, "../.."))
cd(repo_root)
Pkg.activate(".")

try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end

include("src/batchFiles/batchGPU.jl")
include("src/commonBatchs.jl")
using .commonBatchs, UnPack, Symbolics
include("src/flexOPT.jl")
using .flexOPT


## 2. Public Order To Internal Mode


In [ ]:
orders = [-1, 0, 1, 2]
[Main.flexOPT.bspline_order_spec(o) for o in orders]


## 3. Construct Indicator And B-Spline Families


In [ ]:
allNodes = collect(1.0:4.0)
idx_nodesNum = collect(1:4)
idx_refPoints = collect(1:4)
idx_selectedPoints = [2]

params_indicator = (maximumOrder=-1, allNodes=allNodes, idx_nodesNum=idx_nodesNum, idx_refPoints=idx_refPoints, idx_selectedPoints=idx_selectedPoints)
params_bspline = (maximumOrder=1, allNodes=allNodes, idx_nodesNum=idx_nodesNum, idx_refPoints=idx_refPoints, idx_selectedPoints=idx_selectedPoints)

Y_indicator = Main.flexOPT.constructBsplineFamily(params_indicator)
Y_bspline = Main.flexOPT.constructBsplineFamily(params_bspline)

family_summary(f) = (kind=f.kind, order=f.order, has_b=f.b !== nothing)
family_summary(Y_indicator), family_summary(Y_bspline)


## 4. WYYKK Integral With Y = -1


In [ ]:
params_indicator_WYYKK = Dict{String,Any}(
    "orderBspline1D" => 1,
    "YorderBspline1Dμᶜ" => -1,
    "YorderBspline1Dμ" => -1,
    "μᶜs" => [2.0],
    "μs" => [2.0],
    "maxNode" => 4,
    "ν" => 2.0,
    "lᶜ_nᶜ_max" => 2,
    "l_n_max" => 2,
    "ImakeReport" => false,
)

out_indicator = Main.flexOPT.WYYKKIntegralPureSymbolic(params_indicator_WYYKK)
WYYKK_indicator = out_indicator["WYYKK_integral"]
(size(WYYKK_indicator.data), WYYKK_indicator.nodes[1:min(end, 5)])


## 5. Numerical Coefficients With Y = -1


In [ ]:
params_indicator_num = copy(params_indicator_WYYKK)
params_indicator_num["Δ"] = 1.0
coef_indicator = Main.flexOPT.WYYKKIntegralNumerical(params_indicator_num; ImakeReport=false)
coef_indicator[:, :, 1, 1, 1]


## 6. Same Integral With Y = 1


In [ ]:
params_bspline_WYYKK = copy(params_indicator_WYYKK)
params_bspline_WYYKK["YorderBspline1Dμᶜ"] = 1
params_bspline_WYYKK["YorderBspline1Dμ"] = 1
params_bspline_WYYKK["Δ"] = 1.0
coef_bspline = Main.flexOPT.WYYKKIntegralNumerical(params_bspline_WYYKK; ImakeReport=false)
coef_bspline[:, :, 1, 1, 1]


## 7. Optional Report Files


In [ ]:
# Set ImakeReport=true if you want PDF/TXT report files under tmp/WYYKKIntegralPureSymbolic_report.
# For Y=-1, the report writes a TXT marker for the indicator family instead of trying to plot a non-existing B-spline.
params_report = copy(params_indicator_WYYKK)
params_report["ImakeReport"] = true
# out_report = Main.flexOPT.WYYKKIntegralPureSymbolic(params_report)
# out_report["reportDir"]
